# Data Preparation — Close-to-Expiration Markdown Optimisation

This notebook documents the data preparation performed for the **Close-to-Expiration Markdown Optimisation** project.

The workflow covers:
- Store-level data cleaning and missing-value treatment
- Product/label data cleaning and standardisation
- Feature engineering
- Outlier and inconsistency handling
- Preparation of variables used in downstream analysis and machine learning

> **Data availability:** The original datasets are not included in this public repository. To run the notebook locally, place the required input files in the `data/` folder using the filenames referenced below.


## 1. Store Data Preparation


### Load the Store Dataset


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

DATA_DIR = Path("../data")

STORE_FILE = DATA_DIR / "Data_store.xlsx"
LABELS_FILE = DATA_DIR / "Data_labels (1).xlsx"
PRODUCTS_CLEANED_FILE = DATA_DIR / "Products Cleaned Dataset-2.xlsx"

df_store = pd.read_excel(STORE_FILE)
df_store.info()


In [ ]:
print(df_store.groupby('type')['selling_square_ft'].describe())


The distribution of `selling_square_ft` was reviewed by store type to determine an appropriate strategy for missing-value imputation. The similarity between the mean and median suggested that median-based imputation would have limited sensitivity to extreme values.


In [ ]:
print(f"Missing values in 'selling_square_ft': {df_store['selling_square_ft'].isna().sum()}")


In [ ]:
df_store['selling_square_ft'] = (
    df_store.groupby('type')['selling_square_ft']
    .transform(lambda x: x.fillna(x.median()))
)


In [ ]:
print(f"Remaining missing values in 'selling_square_ft': {df_store['selling_square_ft'].isna().sum()}")


In [ ]:
display(df_store[df_store['selling_square_ft'].isna()])


One store remained without a value after the initial imputation. Because store area can be influenced by local geography, population density and property costs, the remaining value was imputed using the median area of stores in the **Vila Real** district.

The missing store type was also filled using the mode of stores in Vila Real.


In [ ]:
df_store['type'] = df_store['type'].replace({0: np.nan, '0': np.nan})

vila_real_area_median = df_store.loc[
    df_store['district'].eq('Vila Real'),
    'selling_square_ft'
].median()

vila_real_type_mode = df_store.loc[
    df_store['district'].eq('Vila Real'),
    'type'
].mode().iloc[0]

df_store['selling_square_ft'] = df_store['selling_square_ft'].fillna(vila_real_area_median)
df_store['type'] = df_store['type'].fillna(vila_real_type_mode)

display(df_store[df_store['idstore'] == 194])


## 2. Product / Label Data Preparation

The product/label dataset contains one row per labelled SKU. The preparation steps below standardise prices, discounts, dates and product attributes, and create features used in later analysis.


### Load the Product / Label Dataset


In [ ]:
df_products = pd.read_excel(LABELS_FILE, decimal=',')
df_products.info()


### Split Price and Discount Values


In [ ]:
df_products[['new_pvp', 'discount']] = df_products['new_pvp (discount)'].str.extract(
    r'([\d.,]+)\s*\(([\d.,%]+)'
)

df_products['new_pvp'] = (
    df_products['new_pvp']
    .str.replace(',', '.', regex=False)
    .astype(float)
)

df_products['discount'] = (
    df_products['discount']
    .str.replace('%', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

new_pvp = df_products.pop('new_pvp')
discount = df_products.pop('discount')

idx = df_products.columns.get_loc('new_pvp (discount)')
df_products.insert(idx + 1, 'new_pvp', new_pvp)
df_products.insert(idx + 2, 'discount', discount)

df_products = df_products.drop(columns=['new_pvp (discount)'])


### Standardise Discount Values


In [ ]:
df_products.loc[df_products['discount'] >= 1, 'discount'] /= 100


### Standardise Date Columns


In [ ]:
date_columns = ['expiring_date', 'labelling_date', 'sell_date']

for col in date_columns:
    df_products[col] = pd.to_datetime(
        df_products[col],
        format='mixed',
        dayfirst=True,
        errors='coerce'
    )

display(df_products[date_columns].head())


### Create `days_until_expiration`


In [ ]:
df_products['days_until_expiration'] = (
    df_products['expiring_date'] - df_products['labelling_date']
).dt.days


### Standardise Brand Values


In [ ]:
df_products['brand'] = df_products['brand'].str.extract(r'(\d+)').astype(int)


### Standardise Numeric Variables


In [ ]:
df_products['sold'] = df_products['sold'].astype('Int64')

df_products['weight (g)'] = pd.to_numeric(
    df_products['weight (g)'],
    errors='coerce'
).astype('Int64')

df_products.info()


## 3. Prepare the Dataset for Sales and Outlier Analysis

The cleaned product dataset is loaded for the subsequent validation and feature-engineering steps.


In [ ]:
df_products = pd.read_excel(PRODUCTS_CLEANED_FILE)
df_products.info()


### Inspect `oldpvp` Outliers


In [ ]:
Q1 = df_products['oldpvp'].quantile(0.25)
Q3 = df_products['oldpvp'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_oldpvp = df_products[
    (df_products['oldpvp'] < lower_bound) |
    (df_products['oldpvp'] > upper_bound)
]

print(f"Number of outliers in 'oldpvp': {len(outliers_oldpvp)}")
print(f"Lower bound: {lower_bound}")
print(f"Upper bound: {upper_bound}")


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x=df_products['oldpvp'])
plt.title('Box Plot of oldpvp')
plt.xlabel('oldpvp')
plt.show()


The identified values were reviewed against the business context. The values `999`, `500` and `46` were treated as erroneous `oldpvp` values in this project and removed.


In [ ]:
df_products = df_products[~df_products['oldpvp'].isin([999, 500, 46])]

print(f"Number of rows after removing known erroneous 'oldpvp' values: {len(df_products)}")
display(df_products.head())


### Inspect `new_pvp` Outliers


In [ ]:
Q1 = df_products['new_pvp'].quantile(0.25)
Q3 = df_products['new_pvp'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_newpvp = df_products[
    (df_products['new_pvp'] < lower_bound) |
    (df_products['new_pvp'] > upper_bound)
]

print(f"Number of outliers in 'new_pvp': {len(outliers_newpvp)}")
print(f"Lower bound: {lower_bound}")
print(f"Upper bound: {upper_bound}")


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x=df_products['new_pvp'])
plt.title('Box Plot of new_pvp')
plt.xlabel('new_pvp')
plt.show()


The `new_pvp` outliers were inspected but retained because they were considered potentially valid product-price observations rather than confirmed data errors.


### Handle Missing `oldpvp` Values


In [ ]:
df_products.dropna(subset=['oldpvp'], inplace=True)

print(f"Number of rows after dropping nulls in 'oldpvp': {len(df_products)}")
display(df_products.isnull().sum())


### Create `days_to_sell`


In [ ]:
df_products['days_to_sell'] = (
    df_products['sell_date'] - df_products['labelling_date']
).dt.days

sold_idx = df_products.columns.get_loc('sold')
days_to_sell = df_products.pop('days_to_sell')
df_products.insert(sold_idx, 'days_to_sell', days_to_sell)

df_products.info()


Some `days_to_sell` values were negative. Since labelling should occur before the sale, these records were treated as inconsistencies.


In [ ]:
min_days_to_sell = df_products['days_to_sell'].min()
max_days_to_sell = df_products['days_to_sell'].max()

print(f"Minimum days to sell: {min_days_to_sell}")
print(f"Maximum days to sell: {max_days_to_sell}")


In [ ]:
negative_days_to_sell = df_products[df_products['days_to_sell'] < 0]

print(
    f"Number of rows with negative 'days_to_sell': "
    f"{len(negative_days_to_sell)}"
)


In [ ]:
print(f"Rows before removing negative 'days_to_sell': {len(df_products)}")

df_products = df_products[df_products['days_to_sell'] >= 0]

print(f"Rows after removing negative 'days_to_sell': {len(df_products)}")
display(df_products.isnull().sum())


### Final Data-Type and Missing-Value Checks


In [ ]:
df_products['days_to_sell'] = df_products['days_to_sell'].astype('Int64')

print(f"Rows before dropping NaNs in 'days_to_sell': {len(df_products)}")
df_products.dropna(subset=['days_to_sell'], inplace=True)
print(f"Rows after dropping NaNs in 'days_to_sell': {len(df_products)}")

display(df_products.isnull().sum())
df_products.info()
